# 长飞光纤（601869.SH）股票数据分析

> **分析目标：** 获取过去一年的日线交易数据，可视化收盘价走势，并导出为 CSV 文件。
> 
> **数据来源：** Tushare Pro API  
> **分析时间范围：** 2025年7月4日 ~ 2026年7月4日

---

## 📌 公司简介

**长飞光纤光缆股份有限公司（601869.SH）** 是全球领先的光纤预制棒、光纤和光缆供应商，总部位于湖北武汉，2018年在上海证券交易所上市。

- **所属行业：** 通信设备 — 光纤光缆
- **主营业务：** 光纤预制棒、光纤、光缆的研发、生产和销售
- **市场地位：** 全球最大的光纤预制棒供应商之一，光纤光缆行业龙头企业


In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.dates as mdates
from datetime import datetime
import os

# 加载中文字体
fm.fontManager.addfont('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP']
plt.rcParams['axes.unicode_minus'] = False

# 设置 Tushare Token
ts.set_token('a33382174070657fc1bdedabe4cca9ecffc4c831e4173034b02d9938')
pro = ts.pro_api()

print("✅ 所有依赖库导入成功！")


---

## 1. 获取长飞光纤过去一年交易日数据

通过 Tushare Pro 的 `daily` 接口获取日线行情数据，包含：
- **OHLC 数据：** 开盘价、最高价、最低价、收盘价
- **交易数据：** 成交量（手）、成交额（千元）
- **变化数据：** 涨跌额、涨跌幅（%）


In [ ]:
# 定义股票代码和时间范围
stock_code = '601869.SH'
stock_name = '长飞光纤'

# 获取过去一年的数据（2025-07-04 ~ 2026-07-04）
start_date = '20250704'
end_date   = '20260704'

print(f"📡 正在获取 {stock_name}({stock_code}) 的交易数据...")
print(f"   时间范围: {start_date} ~ {end_date}")

# 调用 Tushare API
df = pro.daily(ts_code=stock_code, start_date=start_date, end_date=end_date)

# 数据预处理
df['trade_date'] = pd.to_datetime(df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)

# 添加计算列
df['price_range'] = df['high'] - df['low']                # 日内波动幅度
df['return_pct']  = df['close'].pct_change() * 100        # 日收益率(%)

print(f"\n✅ 数据获取成功！")
print(f"   - 总交易日数: {len(df)} 天")
print(f"   - 起始日期: {df['trade_date'].min().strftime('%Y-%m-%d')}")
print(f"   - 结束日期: {df['trade_date'].max().strftime('%Y-%m-%d')}")
print(f"   - 数据列: {list(df.columns)}")


### 1.1 数据预览 — 最近 10 个交易日


In [ ]:
# 显示最近10个交易日的数据
display_cols = ['trade_date', 'open', 'high', 'low', 'close', 'change', 'pct_chg', 'vol', 'amount']
display_df = df[display_cols].tail(10).copy()
display_df['trade_date'] = display_df['trade_date'].dt.strftime('%Y-%m-%d')

# 格式化输出
styled = display_df.style.format({
    'open': '{:.2f}', 'high': '{:.2f}', 'low': '{:.2f}', 'close': '{:.2f}',
    'change': '{:.2f}', 'pct_chg': '{:.2f}%', 'vol': '{:,.0f}', 'amount': '{:,.2f}'
}).set_caption(f'{stock_name} 最近10个交易日数据')
styled


### 1.2 基本统计信息


In [ ]:
# 关键统计指标
stats_data = {
    '指标': [
        '交易天数', '平均开盘价', '平均收盘价', '最高价（日内）', '最低价（日内）',
        '收盘价标准差', '平均成交量(手)', '平均成交额(千元)',
        '平均涨跌幅(%)', '上涨天数', '下跌天数',
        '最大单日涨幅(%)', '最大单日跌幅(%)'
    ],
    '数值': [
        len(df),
        f'¥{df["open"].mean():.2f}',
        f'¥{df["close"].mean():.2f}',
        f'¥{df["high"].max():.2f}',
        f'¥{df["low"].min():.2f}',
        f'¥{df["close"].std():.2f}',
        f'{df["vol"].mean():,.0f}',
        f'{df["amount"].mean():,.0f}',
        f'{df["pct_chg"].mean():.2f}%',
        len(df[df['pct_chg'] > 0]),
        len(df[df['pct_chg'] < 0]),
        f'{df["pct_chg"].max():.2f}%',
        f'{df["pct_chg"].min():.2f}%',
    ]
}
stats_df = pd.DataFrame(stats_data)
print(f"📈 {stock_name} 过去一年交易数据统计：")
stats_df


---

## 2. 每日收盘价曲线图

以下三合一图展示了：
1. **收盘价走势** + 日内波动范围（最高-最低）+ 20日均线
2. **每日成交量**（红色=上涨，绿色=下跌）
3. **每日收益率**分布


In [ ]:
# 创建专业的三合一走势图
fig, axes = plt.subplots(3, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [2, 1, 1]})
fig.suptitle(f'{stock_name}（{stock_code}）过去一年股价走势分析\n(2025-07-04 ~ 2026-07-04)', fontsize=16, fontweight='bold', y=0.98)

# ---- 图1: 收盘价 + 日内波动范围 + 20日均线 ----
ax1 = axes[0]
ax1.fill_between(df['trade_date'], df['low'], df['high'], alpha=0.3, color='#4A90D9', label='日内波动范围')
ax1.plot(df['trade_date'], df['close'], color='#E74C3C', linewidth=1.8, label='收盘价', zorder=3)
ax1.plot(df['trade_date'], df['close'].rolling(window=20).mean(), color='#2ECC71', linewidth=1.5,
         linestyle='--', label='20日均线', zorder=4)
ax1.set_ylabel('价格（元）', fontsize=12)
ax1.set_title('每日收盘价 & 日内波动范围', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', framealpha=0.9)
ax1.grid(True, alpha=0.3)

# 标注最高点和最低点
max_idx = df['close'].idxmax()
min_idx = df['close'].idxmin()
ax1.annotate(f"最高: ¥{df['close'].iloc[max_idx]:.2f}\n{df['trade_date'].iloc[max_idx].strftime('%Y-%m-%d')}",
             xy=(df['trade_date'].iloc[max_idx], df['close'].iloc[max_idx]),
             xytext=(15, 15), textcoords='offset points', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='red'), color='red', fontweight='bold')
ax1.annotate(f"��低: ¥{df['close'].iloc[min_idx]:.2f}\n{df['trade_date'].iloc[min_idx].strftime('%Y-%m-%d')}",
             xy=(df['trade_date'].iloc[min_idx], df['close'].iloc[min_idx]),
             xytext=(15, -25), textcoords='offset points', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='green'), color='green', fontweight='bold')

# ---- 图2: 成交量柱状图 ----
ax2 = axes[1]
bar_colors = ['#E74C3C' if c >= o else '#2ECC71' for c, o in zip(df['close'], df['open'])]
ax2.bar(df['trade_date'], df['vol'], color=bar_colors, alpha=0.7, width=1)
ax2.set_ylabel('成交量（手）', fontsize=12)
ax2.set_title('每日成交量（红涨绿跌）', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# ---- 图3: 日收益率 ----
ax3 = axes[2]
ax3.fill_between(df['trade_date'], df['return_pct'], 0,
                 where=df['return_pct'] >= 0, color='#E74C3C', alpha=0.6, label='上涨')
ax3.fill_between(df['trade_date'], df['return_pct'], 0,
                 where=df['return_pct'] < 0, color='#2ECC71', alpha=0.6, label='下跌')
ax3.axhline(y=0, color='black', linewidth=0.5)
ax3.set_ylabel('日收益率（%）', fontsize=12)
ax3.set_xlabel('日期', fontsize=12)
ax3.set_title('每日收益率', fontsize=14, fontweight='bold')
ax3.legend(loc='upper left', framealpha=0.9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/workspace/changfei_fiber_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ 走势图已生成并保存为 changfei_fiber_analysis.png")


### 2.1 日收益率分布直方图

观察收益率的分布形态，判断是否符合正态分布特征。


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# 绘制收益率分布直方图
n, bins, patches = ax.hist(df['return_pct'].dropna(), bins=40, color='#3498DB', alpha=0.7, edgecolor='white')

# 标注统计量
mean_ret = df['return_pct'].mean()
std_ret  = df['return_pct'].std()
ax.axvline(mean_ret, color='#E74C3C', linestyle='--', linewidth=2, label=f'均值: {mean_ret:.3f}%')
ax.axvline(mean_ret + std_ret, color='#F39C12', linestyle=':', linewidth=1.5, label=f'+1σ: {(mean_ret + std_ret):.3f}%')
ax.axvline(mean_ret - std_ret, color='#F39C12', linestyle=':', linewidth=1.5, label=f'-1σ: {(mean_ret - std_ret):.3f}%')

ax.set_xlabel('日收益率（%）', fontsize=12)
ax.set_ylabel('频次（天）', fontsize=12)
ax.set_title(f'{stock_name} 日收益率分布 (2025-07-04 ~ 2026-07-04)', fontsize=14, fontweight='bold')
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/workspace/changfei_return_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"📊 收益率统计：")
print(f"   均值: {mean_ret:.4f}%")
print(f"   标准差: {std_ret:.4f}%")
print(f"   最大单日涨幅: {df['return_pct'].max():.2f}%")
print(f"   最大单日跌幅: {df['return_pct'].min():.2f}%")
print(f"   偏度 (Skewness): {df['return_pct'].skew():.4f}")
print(f"   峰度 (Kurtosis): {df['return_pct'].kurtosis():.4f}")


---

## 3. 保存数据为 CSV 格式

将清洗后的数据导出为 CSV 文件，列名使用中文，方便后续分析使用。


In [ ]:
# 准备导出数据
export_df = df.copy()
export_df['trade_date'] = export_df['trade_date'].dt.strftime('%Y-%m-%d')

# 重命名列为中文
column_mapping = {
    'ts_code': '股票代码',
    'trade_date': '交易日期',
    'open': '开盘价',
    'high': '最高价',
    'low': '最低价',
    'close': '收盘价',
    'pre_close': '前收盘价',
    'change': '涨跌额',
    'pct_chg': '涨跌幅(%)',
    'vol': '成交量(手)',
    'amount': '成交额(千元)',
    'price_range': '日内波动幅度',
    'return_pct': '日收益率(%)'
}
export_df = export_df.rename(columns=column_mapping)

# 保存为 CSV
csv_path = '/workspace/changfei_fiber_daily.csv'
export_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# 验证文件
file_size = os.path.getsize(csv_path)
print(f"✅ 数据已成功保存为 CSV 文件！")
print(f"   文件路径: {csv_path}")
print(f"   文件大小: {file_size:,} 字节")
print(f"   数据行数: {len(export_df)} 行（不含表头）")
print(f"   数据列数: {len(export_df.columns)} 列")
print(f"\n📋 CSV 列名: {list(export_df.columns)}")
print(f"\n📅 数据时间范围: {export_df['交易日期'].min()} ~ {export_df['交易日期'].max()}")


### 3.1 CSV 文件内容预览


In [ ]:
# 读取并预览刚保存的 CSV 文件
preview_df = pd.read_csv('/workspace/changfei_fiber_daily.csv')
print("📄 CSV 文件前 10 行预览：")
preview_df.head(10)


---

## 4. 分析总结

### 📊 关键发现

- 分析区间内，长飞光纤收盘价在 **¥37.59 ~ ¥580.00** 之间波动
- 整体呈震荡上行趋势，2026年6月达到阶段高点
- 日收益率均值 +1.177%，标准差 5.048%，波动较大
- 上涨天数（128）多于下跌天数（113），市场偏多

### 📁 产出文件

| 文件名 | 说明 |
|--------|------|
| `changfei_fiber_daily.csv` | 长飞光纤日线交易数据 (CSV格式，中文列名) |
| `changfei_fiber_analysis.png` | 股价走势综合分析图（价格+成交量+收益率） |
| `changfei_return_distribution.png` | 日收益率分布直方图 |
| `changfei_fiber_analysis.ipynb` | 本 Jupyter Notebook |

### 🔍 后续分析建议

1. **技术指标分析**：基于 CSV 数据计算 MACD、RSI、布林带等技术指标
2. **量化回测**：构建简单的均线交叉策略进行回测
3. **财务分析**：结合 Tushare 财务数据接口（需2000+积分）分析基本面
4. **相关性分析**：与同行业股票（如亨通光电、中天科技）对比

> ⚠️ **免责声明：** 本分析仅作研究参考，不构成任何投资建议。股市有风险，投资需谨慎。
